# miRNA Periodontal Disease Analysis: Comprehensive Jupyter Notebook

## Project Overview
This notebook performs comprehensive analysis of miRNA expression data to identify biomarkers for periodontal disease progression (Healthy → Gingivitis → Periodontitis).

**Key Features:**
- ΔΔCt transformation pipeline with GAPDH reference normalization
- Statistical analysis with FDR correction and effect size calculations
- Machine learning classification models with cross-validation
- Dimensionality reduction and clustering validation
- Comprehensive visualization suite with organized outputs

**Authors:** AI-driven Analytical Scientists  
**Date:** July 16, 2025  
**Dataset:** miRNA-saliva-qPCR-results.csv (103 samples, 14 variables)

---

## Analysis Workflow
1. **Environment Setup** - Library imports and configuration
2. **Data Loading & Preprocessing** - Data quality checks and transformations
3. **Exploratory Data Analysis** - Descriptive statistics and initial insights
4. **Statistical Analysis** - Hypothesis testing and differential expression
5. **Machine Learning Models** - Classification and predictive modeling
6. **Dimensionality Reduction** - PCA, t-SNE, UMAP analysis
7. **Visualization Generation** - Comprehensive plotting suite
8. **Results Export** - Organized output structure with Title Case naming

# 1. Environment Setup and Configuration

Setting up the analysis environment with all required libraries and output directory structure.

In [ ]:
#!/usr/bin/env python3
"""
miRNA Periodontal Disease Analysis - Comprehensive Jupyter Notebook
================================================================

This notebook performs comprehensive analysis of miRNA expression data to identify
biomarkers for periodontal disease progression (Healthy → Gingivitis → Periodontitis).
"""

# Core scientific computing
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# Statistical analysis
from scipy import stats
from scipy.stats import (
    levene,
    shapiro,
    kruskal,
    mannwhitneyu,
    ttest_ind,
    chi2_contingency,
    pearsonr,
    spearmanr,
)
import statsmodels.stats.multitest as smt

# Machine learning
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
    accuracy_score,
)
from sklearn.inspection import permutation_importance, partial_dependence
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# Dimensionality reduction
import umap

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle

# Additional utilities
import os
import logging
from datetime import datetime
import json

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.FileHandler("analysis.log"), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

# Set display options
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

# Configure matplotlib and seaborn
plt.style.use("default")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["legend.fontsize"] = 10

print("✅ Environment setup complete!")
print(f"📊 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🐍 Python version: {sys.version}")
print(f"📊 Pandas version: {pd.__version__}")
print(f"📈 NumPy version: {np.__version__}")

import sys


In [ ]:
# Configure output directory structure
BASE_OUTPUT_DIR = "outputs/jupyter_notebook"
OUTPUT_DIRS = {
    "base": BASE_OUTPUT_DIR,
    "plots": f"{BASE_OUTPUT_DIR}/plots",
    "tables": f"{BASE_OUTPUT_DIR}/tables",
    "sensitivity": f"{BASE_OUTPUT_DIR}/sensitivity",
}

# Create output directories
for dir_name, dir_path in OUTPUT_DIRS.items():
    os.makedirs(dir_path, exist_ok=True)
    print(f"📁 Created directory: {dir_path}")


# Define file naming conventions
def get_output_path(filename, output_type="plots"):
    """Get standardized output path with Title Case naming"""
    if not filename.endswith((".png", ".jpg", ".jpeg", ".pdf", ".csv", ".txt")):
        filename += ".png"  # Default to PNG for plots

    return os.path.join(OUTPUT_DIRS[output_type], filename)


# Analysis configuration
ANALYSIS_CONFIG = {
    "random_state": 42,
    "test_size": 0.2,
    "cv_folds": 5,
    "alpha_level": 0.05,
    "fdr_method": "fdr_bh",
    "effect_size_threshold": 1.0,
    "groups": ["S", "G", "P"],  # Healthy, Gingivitis, Periodontitis
    "mirna_targets": ["mir146a", "mir146b", "mir155", "mir203", "mir223", "mir381p"],
    "clinical_vars": [
        "plaque_index",
        "gingival_index",
        "pocket_depth",
        "bleeding_on_probing",
        "number_of_missing_teeth",
    ],
    "demographic_vars": ["AGE", "SEX"],
}

print("🔧 Configuration complete!")
print(f"📊 Analysis parameters: {json.dumps(ANALYSIS_CONFIG, indent=2)}")


# 2. Data Loading and Preprocessing

Loading the miRNA-saliva-qPCR dataset and performing essential data quality checks and transformations.

In [ ]:
# Load the dataset
DATA_FILE = "miRNA-saliva-qPCR-results.csv"
logger.info(f"Loading data from {DATA_FILE}")

try:
    df = pd.read_csv(DATA_FILE)
    logger.info("✅ Data loaded successfully")
except FileNotFoundError:
    logger.error(f"❌ File {DATA_FILE} not found!")
    raise
except Exception as e:
    logger.error(f"❌ Error loading data: {e}")
    raise

# Data validation and quality checks
print("📊 DATASET OVERVIEW")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Groups: {df['GROUP'].value_counts().to_dict()}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

# Check for missing values
print("\n🔍 DATA QUALITY ASSESSMENT")
print("=" * 50)
missing_data = df.isnull().sum()
if missing_data.any():
    print("Missing values detected:")
    print(missing_data[missing_data > 0])
else:
    print("✅ No missing values detected")

# Data types and basic statistics
print("\n📋 DATA TYPES")
print("=" * 50)
print(df.dtypes)

print("\n📊 BASIC STATISTICS")
print("=" * 50)
print(df.describe())

# Validate group codes
expected_groups = {"S", "G", "P"}
actual_groups = set(df["GROUP"].unique())
if actual_groups != expected_groups:
    logger.warning(
        f"⚠️  Group codes mismatch. Expected: {expected_groups}, Found: {actual_groups}"
    )
else:
    logger.info("✅ Group codes validated")

# Display first few rows
print("\n📖 SAMPLE DATA")
print("=" * 50)
print(df.head())


In [ ]:
# ΔΔCt Transformation Pipeline
print("🧬 PERFORMING ΔΔCt TRANSFORMATION")
print("=" * 50)

# Step 1: Calculate ΔCt (Ct_miRNA - Ct_GAPDH)
print("Step 1: Calculating ΔCt values...")
for mirna in ANALYSIS_CONFIG["mirna_targets"]:
    dct_col = f"dCt_{mirna}"
    df[dct_col] = df[mirna] - df["GAPDH"]
    print(f"  ✓ {dct_col} = {mirna} - GAPDH")

# Step 2: Calculate calibrator values (mean ΔCt for Healthy group)
print("\nStep 2: Calculating calibrator values (Healthy group means)...")
healthy_group = df[df["GROUP"] == "S"]
calibrators = {}

for mirna in ANALYSIS_CONFIG["mirna_targets"]:
    dct_col = f"dCt_{mirna}"
    calibrator_value = healthy_group[dct_col].mean()
    calibrators[mirna] = calibrator_value
    print(f"  ✓ Calibrator for {mirna}: {calibrator_value:.3f}")

# Step 3: Calculate ΔΔCt (ΔCt_sample - ΔCt_calibrator)
print("\nStep 3: Calculating ΔΔCt values...")
for mirna in ANALYSIS_CONFIG["mirna_targets"]:
    dct_col = f"dCt_{mirna}"
    ddct_col = f"ddCt_{mirna}"
    df[ddct_col] = df[dct_col] - calibrators[mirna]
    print(f"  ✓ {ddct_col} = {dct_col} - {calibrators[mirna]:.3f}")

# Step 4: Calculate RQ values (2^(-ΔΔCt))
print("\nStep 4: Calculating RQ values (2^(-ΔΔCt))...")
for mirna in ANALYSIS_CONFIG["mirna_targets"]:
    ddct_col = f"ddCt_{mirna}"
    rq_col = f"RQ_{mirna}"
    df[rq_col] = 2 ** (-df[ddct_col])
    print(f"  ✓ {rq_col} = 2^(-{ddct_col})")

# Save calibrator values
calibrator_df = pd.DataFrame([calibrators]).T
calibrator_df.columns = ["Calibrator_Value"]
calibrator_df.index.name = "miRNA"
calibrator_df.to_csv(get_output_path("Calibration_Table.csv", "tables"))

print("✅ ΔΔCt transformation complete!")
print(
    f"📊 Calibrator values saved to: {get_output_path('Calibration_Table.csv', 'tables')}"
)

# Display transformation results
print("\n📊 TRANSFORMATION RESULTS PREVIEW")
print("=" * 50)
transformation_cols = ["GROUP"] + [
    f"RQ_{mirna}" for mirna in ANALYSIS_CONFIG["mirna_targets"]
]
print(df[transformation_cols].head(10))


In [ ]:
# Reference Gene Validation (GAPDH Stability)
print("🔬 REFERENCE GENE VALIDATION")
print("=" * 50)

# Test GAPDH stability across groups
gapdh_by_group = df.groupby("GROUP")["GAPDH"].agg(["mean", "std", "count"])
print("GAPDH stability by group:")
print(gapdh_by_group)

# Statistical test for GAPDH stability
groups = [
    df[df["GROUP"] == group]["GAPDH"].values for group in ANALYSIS_CONFIG["groups"]
]
kruskal_stat, kruskal_p = kruskal(*groups)
print(f"\nKruskal-Wallis test for GAPDH stability:")
print(f"H-statistic: {kruskal_stat:.3f}")
print(f"p-value: {kruskal_p:.3f}")

if kruskal_p < 0.05:
    print("⚠️  GAPDH shows significant variation across groups - proceed with caution!")
else:
    print("✅ GAPDH is stable across groups")

# Create GAPDH stability plot
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="GROUP", y="GAPDH", palette="Set2")
plt.title("GAPDH Stability Across Groups")
plt.ylabel("GAPDH Ct Value")
plt.xlabel("Group")
plt.tight_layout()
plt.savefig(
    get_output_path("GAPDH_Stability_Boxplot.png"), dpi=300, bbox_inches="tight"
)
plt.show()

# Calculate correlations between GAPDH and clinical variables
print("\n🔍 GAPDH vs Clinical Variables Correlations:")
gapdh_correlations = []
for clinical_var in ANALYSIS_CONFIG["clinical_vars"]:
    if clinical_var in df.columns:
        corr, p_val = pearsonr(df["GAPDH"], df[clinical_var])
        gapdh_correlations.append(
            {"Clinical_Variable": clinical_var, "Correlation": corr, "P_Value": p_val}
        )
        print(f"  {clinical_var}: r={corr:.3f}, p={p_val:.3f}")

# Save GAPDH correlations
gapdh_corr_df = pd.DataFrame(gapdh_correlations)
gapdh_corr_df.to_csv(
    get_output_path("GAPDH_Clinical_Correlations.csv", "tables"), index=False
)

print(
    f"📊 GAPDH analysis saved to: {get_output_path('GAPDH_Clinical_Correlations.csv', 'tables')}"
)

# Document reference gene limitation
limitation_report = f"""
REFERENCE GENE LIMITATION REPORT
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

Single Reference Gene Analysis:
- Reference gene: GAPDH
- Stability across groups: {'STABLE' if kruskal_p >= 0.05 else 'UNSTABLE'}
- Kruskal-Wallis p-value: {kruskal_p:.6f}

Limitations:
1. Single reference gene may not capture all sources of technical variation
2. Multiple reference genes are recommended for robust normalization
3. Results should be interpreted considering potential reference gene bias
4. Future validation with additional reference genes is recommended

Recommendation: Consider validating key findings with alternative reference genes
"""

with open(get_output_path("Reference_Gene_Limitation_Report.txt", "tables"), "w") as f:
    f.write(limitation_report)

print("📋 Reference gene limitation report saved")


# 3. Exploratory Data Analysis

Comprehensive exploratory analysis including descriptive statistics, correlations, and initial visualizations.

In [ ]:
# Demographic and Clinical Statistics
print("👥 DEMOGRAPHIC AND CLINICAL STATISTICS")
print("=" * 50)

# Group-wise statistics
demographic_stats = []

for group in ANALYSIS_CONFIG["groups"]:
    group_data = df[df["GROUP"] == group]

    # Basic demographics
    stats_row = {
        "Group": group,
        "N": len(group_data),
        "Age_Mean": group_data["AGE"].mean(),
        "Age_Std": group_data["AGE"].std(),
        "Sex_M_Count": (group_data["SEX"] == "M").sum(),
        "Sex_F_Count": (group_data["SEX"] == "F").sum(),
        "Sex_M_Percent": (group_data["SEX"] == "M").mean() * 100,
    }

    # Clinical variables
    for clinical_var in ANALYSIS_CONFIG["clinical_vars"]:
        if clinical_var in group_data.columns:
            stats_row[f"{clinical_var}_Mean"] = group_data[clinical_var].mean()
            stats_row[f"{clinical_var}_Std"] = group_data[clinical_var].std()

    demographic_stats.append(stats_row)

# Create demographic statistics table
demo_stats_df = pd.DataFrame(demographic_stats)
demo_stats_df.to_csv(
    get_output_path("Demographic_Clinical_Stats.csv", "tables"), index=False
)

print("📊 Demographic and Clinical Statistics by Group:")
print(demo_stats_df.round(3))

# Normality testing for continuous variables
print("\n🔬 NORMALITY TESTING")
print("=" * 50)

normality_results = []
continuous_vars = (
    ["AGE"]
    + ANALYSIS_CONFIG["clinical_vars"]
    + [f"RQ_{mirna}" for mirna in ANALYSIS_CONFIG["mirna_targets"]]
)

for var in continuous_vars:
    if var in df.columns:
        # Shapiro-Wilk test
        stat, p_value = shapiro(df[var])
        normality_results.append(
            {
                "Variable": var,
                "Shapiro_Wilk_Stat": stat,
                "P_Value": p_value,
                "Is_Normal": p_value > 0.05,
            }
        )

        print(
            f"{var}: W={stat:.3f}, p={p_value:.3f} {'(Normal)' if p_value > 0.05 else '(Non-normal)'}"
        )

# Save normality test results
normality_df = pd.DataFrame(normality_results)
normality_df.to_csv(
    get_output_path("Normality_Test_Results.csv", "tables"), index=False
)

print(
    f"\n📊 Normality test results saved to: {get_output_path('Normality_Test_Results.csv', 'tables')}"
)

# Overall correlation analysis
print("\n🔗 CORRELATION ANALYSIS")
print("=" * 50)

# Select numeric columns for correlation
numeric_cols = (
    ["AGE"]
    + ANALYSIS_CONFIG["clinical_vars"]
    + [f"RQ_{mirna}" for mirna in ANALYSIS_CONFIG["mirna_targets"]]
)
numeric_data = df[numeric_cols].select_dtypes(include=[np.number])

# Calculate correlation matrix
correlation_matrix = numeric_data.corr()

# Create correlation heatmap
plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(
    correlation_matrix,
    mask=mask,
    annot=True,
    cmap="coolwarm",
    center=0,
    square=True,
    fmt=".2f",
    cbar_kws={"shrink": 0.8},
)
plt.title("Correlation Matrix: Clinical Variables and miRNA Expression")
plt.tight_layout()
plt.savefig(get_output_path("Correlation_Heatmap.png"), dpi=300, bbox_inches="tight")
plt.show()

# Save correlation matrix
correlation_matrix.to_csv(get_output_path("Overall_Correlations.csv", "tables"))

# Identify significant correlations
significant_correlations = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i + 1, len(correlation_matrix.columns)):
        var1 = correlation_matrix.columns[i]
        var2 = correlation_matrix.columns[j]
        corr_value = correlation_matrix.iloc[i, j]

        if abs(corr_value) > 0.3:  # Threshold for reporting
            significant_correlations.append(
                {
                    "Variable_1": var1,
                    "Variable_2": var2,
                    "Correlation": corr_value,
                    "Abs_Correlation": abs(corr_value),
                }
            )

# Sort by absolute correlation
significant_correlations.sort(key=lambda x: x["Abs_Correlation"], reverse=True)

print("\n🔍 Significant Correlations (|r| > 0.3):")
for corr in significant_correlations[:10]:  # Top 10
    print(f"  {corr['Variable_1']} ↔ {corr['Variable_2']}: r={corr['Correlation']:.3f}")

print(
    f"\n📊 Full correlation matrix saved to: {get_output_path('Overall_Correlations.csv', 'tables')}"
)


In [ ]:
# RQ Distribution Analysis
print("📊 RQ DISTRIBUTION ANALYSIS")
print("=" * 50)

# Create RQ distribution plots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, mirna in enumerate(ANALYSIS_CONFIG["mirna_targets"]):
    rq_col = f"RQ_{mirna}"

    # Distribution by group
    for group in ANALYSIS_CONFIG["groups"]:
        group_data = df[df["GROUP"] == group][rq_col]
        axes[i].hist(group_data, alpha=0.7, label=f"Group {group}", bins=20)

    axes[i].set_title(f"{mirna} Expression Distribution")
    axes[i].set_xlabel("RQ Value")
    axes[i].set_ylabel("Frequency")
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(get_output_path("RQ_Distributions.png"), dpi=300, bbox_inches="tight")
plt.show()

# Summary statistics for RQ values
print("\n📊 RQ VALUE SUMMARY STATISTICS")
print("=" * 50)

rq_cols = [f"RQ_{mirna}" for mirna in ANALYSIS_CONFIG["mirna_targets"]]
rq_summary = df.groupby("GROUP")[rq_cols].agg(["mean", "std", "median", "min", "max"])

print("RQ summary by group:")
print(rq_summary.round(3))

# Clinical variables by group visualization
print("\n📊 CLINICAL VARIABLES BY GROUP")
print("=" * 50)

# Create clinical variables plot
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, clinical_var in enumerate(ANALYSIS_CONFIG["clinical_vars"]):
    if clinical_var in df.columns and i < len(axes):
        sns.boxplot(data=df, x="GROUP", y=clinical_var, ax=axes[i], palette="Set3")
        axes[i].set_title(f'{clinical_var.replace("_", " ").title()} by Group')
        axes[i].set_xlabel("Group")
        axes[i].set_ylabel(clinical_var.replace("_", " ").title())
        axes[i].grid(True, alpha=0.3)

# Hide unused subplots
for i in range(len(ANALYSIS_CONFIG["clinical_vars"]), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.savefig(
    get_output_path("Clinical_Variables_By_Group.png"), dpi=300, bbox_inches="tight"
)
plt.show()

print("✅ Exploratory data analysis complete!")
print(f"📊 {len(significant_correlations)} significant correlations identified")
print(f"📈 Distribution plots saved to: {get_output_path('RQ_Distributions.png')}")
print(
    f"📊 Clinical variables plot saved to: {get_output_path('Clinical_Variables_By_Group.png')}"
)


# 4. Statistical Analysis

Comprehensive statistical analysis with omnibus testing, pairwise comparisons, FDR correction, and effect size calculations.

In [ ]:
# Statistical Analysis Functions
def calculate_effect_size(group1, group2):
    """Calculate Cohen's d effect size"""
    mean1, mean2 = np.mean(group1), np.mean(group2)
    pooled_std = np.sqrt(
        ((np.std(group1, ddof=1) ** 2) + (np.std(group2, ddof=1) ** 2)) / 2
    )
    return (mean1 - mean2) / pooled_std if pooled_std > 0 else 0


def perform_omnibus_test(data, groups, variable):
    """Perform omnibus test (Kruskal-Wallis)"""
    group_data = [data[data["GROUP"] == group][variable].dropna() for group in groups]

    # Remove empty groups
    group_data = [group for group in group_data if len(group) > 0]

    if len(group_data) < 2:
        return None, None, None

    try:
        stat, p_value = kruskal(*group_data)
        return stat, p_value, len(group_data)
    except:
        return None, None, None


def perform_pairwise_comparison(data, var, group1, group2):
    """Perform pairwise comparison between two groups"""
    data1 = data[data["GROUP"] == group1][var].dropna()
    data2 = data[data["GROUP"] == group2][var].dropna()

    if len(data1) == 0 or len(data2) == 0:
        return None, None, None, None, None

    # Mann-Whitney U test (non-parametric)
    try:
        stat, p_value = mannwhitneyu(data1, data2, alternative="two-sided")
        effect_size = calculate_effect_size(data1, data2)
        log2_fc = (
            np.log2(np.median(data1) / np.median(data2)) if np.median(data2) > 0 else 0
        )
        median_diff = np.median(data1) - np.median(data2)

        return stat, p_value, effect_size, log2_fc, median_diff
    except:
        return None, None, None, None, None


# Omnibus Testing
print("🔬 OMNIBUS TESTING (Kruskal-Wallis)")
print("=" * 50)

omnibus_results = []
rq_cols = [f"RQ_{mirna}" for mirna in ANALYSIS_CONFIG["mirna_targets"]]

for variable in rq_cols:
    stat, p_value, n_groups = perform_omnibus_test(
        df, ANALYSIS_CONFIG["groups"], variable
    )

    if stat is not None:
        omnibus_results.append(
            {
                "Variable": variable,
                "Kruskal_Wallis_Stat": stat,
                "P_Value": p_value,
                "N_Groups": n_groups,
                "Significant": p_value < ANALYSIS_CONFIG["alpha_level"],
            }
        )

        print(
            f"{variable}: H={stat:.3f}, p={p_value:.3f} {'***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else ''}"
        )

# Apply FDR correction
omnibus_df = pd.DataFrame(omnibus_results)
if len(omnibus_df) > 0:
    _, omnibus_df["Q_Value"], _, _ = smt.multipletests(
        omnibus_df["P_Value"], method=ANALYSIS_CONFIG["fdr_method"]
    )
    omnibus_df["FDR_Significant"] = (
        omnibus_df["Q_Value"] < ANALYSIS_CONFIG["alpha_level"]
    )

omnibus_df.to_csv(get_output_path("Omnibus_Test_Results.csv", "tables"), index=False)

print(
    f"\n📊 Omnibus test results saved to: {get_output_path('Omnibus_Test_Results.csv', 'tables')}"
)
print(
    f"🔍 {sum(omnibus_df['FDR_Significant'])} miRNAs show significant omnibus effects (FDR < 0.05)"
)

# Display significant omnibus results
significant_omnibus = omnibus_df[omnibus_df["FDR_Significant"]]
if len(significant_omnibus) > 0:
    print("\n✅ Significant omnibus effects (FDR corrected):")
    print(
        significant_omnibus[
            ["Variable", "Kruskal_Wallis_Stat", "P_Value", "Q_Value"]
        ].round(4)
    )


In [ ]:
# Pairwise Comparisons
print("\n🔬 PAIRWISE COMPARISONS")
print("=" * 50)

# Define comparison pairs
comparison_pairs = [
    ("S", "P", "H_vs_P"),  # Healthy vs Periodontitis
    ("S", "G", "H_vs_G"),  # Healthy vs Gingivitis
    ("G", "P", "G_vs_P"),  # Gingivitis vs Periodontitis
]

# Store all pairwise results
all_pairwise_results = {}

for group1, group2, comparison_name in comparison_pairs:
    print(f"\n📊 {comparison_name.replace('_', ' ')} Comparison:")
    print("-" * 30)

    pairwise_results = []

    for variable in rq_cols:
        stat, p_value, effect_size, log2_fc, median_diff = perform_pairwise_comparison(
            df, variable, group1, group2
        )

        if stat is not None:
            pairwise_results.append(
                {
                    "miRNA": variable.replace("RQ_", ""),
                    "Group1": group1,
                    "Group2": group2,
                    "Comparison": comparison_name,
                    "Mann_Whitney_U": stat,
                    "P_Value": p_value,
                    "Log2_FC": log2_fc,
                    "Effect_Size": effect_size,
                    "Median_Diff": median_diff,
                }
            )

            print(
                f"  {variable}: U={stat:.1f}, p={p_value:.3f}, log2FC={log2_fc:.3f}, d={effect_size:.3f}"
            )

    # Create DataFrame and apply FDR correction
    pairwise_df = pd.DataFrame(pairwise_results)
    if len(pairwise_df) > 0:
        _, pairwise_df["Q_Value"], _, _ = smt.multipletests(
            pairwise_df["P_Value"], method=ANALYSIS_CONFIG["fdr_method"]
        )
        pairwise_df["FDR_Significant"] = (
            pairwise_df["Q_Value"] < ANALYSIS_CONFIG["alpha_level"]
        )
        pairwise_df["Effect_Size_Large"] = np.abs(pairwise_df["Effect_Size"]) > 0.8
        pairwise_df["Log2FC_Threshold"] = (
            np.abs(pairwise_df["Log2_FC"]) > ANALYSIS_CONFIG["effect_size_threshold"]
        )

        # Save results
        output_filename = f"{comparison_name}_Results.csv"
        pairwise_df.to_csv(get_output_path(output_filename, "tables"), index=False)
        all_pairwise_results[comparison_name] = pairwise_df

        # Identify candidate biomarkers
        candidates = pairwise_df[
            (pairwise_df["FDR_Significant"]) & (pairwise_df["Log2FC_Threshold"])
        ]

        if len(candidates) > 0:
            print(
                f"  ✅ {len(candidates)} candidate biomarkers identified (FDR < 0.05, |log2FC| > 1)"
            )
            candidate_filename = f"Candidate_Biomarkers_{comparison_name}.csv"
            candidates.to_csv(
                get_output_path(candidate_filename, "tables"), index=False
            )

            # Display top candidates
            candidates_sorted = candidates.sort_values("Q_Value")
            print("  Top candidates:")
            for _, row in candidates_sorted.head(3).iterrows():
                print(
                    f"    {row['miRNA']}: q={row['Q_Value']:.3f}, log2FC={row['Log2_FC']:.3f}"
                )
        else:
            print(f"  ❌ No candidate biomarkers identified")

        print(f"  📊 Results saved to: {get_output_path(output_filename, 'tables')}")

print(f"\n✅ Pairwise comparisons complete!")
print(f"📊 {len(all_pairwise_results)} comparison sets analyzed")


In [ ]:
# Volcano Plots for Differential Expression
print("\n🌋 VOLCANO PLOTS")
print("=" * 50)

# Create volcano plots for each comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, (comparison_name, results_df) in enumerate(all_pairwise_results.items()):
    ax = axes[i]

    # Calculate -log10(p-value)
    results_df["neg_log10_p"] = -np.log10(results_df["P_Value"])

    # Define significance thresholds
    p_threshold = -np.log10(ANALYSIS_CONFIG["alpha_level"])
    fc_threshold = ANALYSIS_CONFIG["effect_size_threshold"]

    # Color points based on significance
    colors = []
    for _, row in results_df.iterrows():
        if row["FDR_Significant"] and abs(row["Log2_FC"]) > fc_threshold:
            colors.append("red")  # Significant
        elif row["FDR_Significant"]:
            colors.append("orange")  # FDR significant but small effect
        elif abs(row["Log2_FC"]) > fc_threshold:
            colors.append("blue")  # Large effect but not significant
        else:
            colors.append("gray")  # Not significant

    # Create scatter plot
    scatter = ax.scatter(
        results_df["Log2_FC"], results_df["neg_log10_p"], c=colors, alpha=0.7, s=60
    )

    # Add threshold lines
    ax.axhline(y=p_threshold, color="black", linestyle="--", alpha=0.5)
    ax.axvline(x=fc_threshold, color="black", linestyle="--", alpha=0.5)
    ax.axvline(x=-fc_threshold, color="black", linestyle="--", alpha=0.5)

    # Annotate significant points
    for _, row in results_df.iterrows():
        if row["FDR_Significant"] and abs(row["Log2_FC"]) > fc_threshold:
            ax.annotate(
                row["miRNA"],
                (row["Log2_FC"], row["neg_log10_p"]),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=8,
            )

    # Formatting
    ax.set_xlabel("Log2 Fold Change")
    ax.set_ylabel("-Log10(P-value)")
    ax.set_title(f'Volcano Plot: {comparison_name.replace("_", " ")}')
    ax.grid(True, alpha=0.3)

    # Add legend
    legend_elements = [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="red",
            markersize=8,
            label="Significant (FDR < 0.05, |log2FC| > 1)",
        ),
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="orange",
            markersize=8,
            label="FDR significant only",
        ),
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="blue",
            markersize=8,
            label="Large effect only",
        ),
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="gray",
            markersize=8,
            label="Not significant",
        ),
    ]
    ax.legend(handles=legend_elements, loc="upper right", fontsize=8)

plt.tight_layout()
plt.savefig(get_output_path("Volcano_Plots.png"), dpi=300, bbox_inches="tight")
plt.show()

# Create summary of all significant results
print("\n📊 DIFFERENTIAL EXPRESSION SUMMARY")
print("=" * 50)

all_significant = []
for comparison_name, results_df in all_pairwise_results.items():
    significant_results = results_df[
        (results_df["FDR_Significant"]) & (results_df["Log2FC_Threshold"])
    ]

    for _, row in significant_results.iterrows():
        all_significant.append(
            {
                "miRNA": row["miRNA"],
                "Comparison": comparison_name,
                "Log2_FC": row["Log2_FC"],
                "P_Value": row["P_Value"],
                "Q_Value": row["Q_Value"],
                "Effect_Size": row["Effect_Size"],
            }
        )

if all_significant:
    significant_df = pd.DataFrame(all_significant)
    significant_df = significant_df.sort_values(["miRNA", "Q_Value"])

    # Save comprehensive significant results
    significant_df.to_csv(
        get_output_path("All_Significant_Results.csv", "tables"), index=False
    )

    print(
        f"✅ {len(all_significant)} significant differential expression results identified"
    )
    print(
        f"📊 Results saved to: {get_output_path('All_Significant_Results.csv', 'tables')}"
    )

    # Display summary by miRNA
    print("\n🔍 Summary by miRNA:")
    for mirna in significant_df["miRNA"].unique():
        mirna_results = significant_df[significant_df["miRNA"] == mirna]
        print(f"  {mirna}: {len(mirna_results)} significant comparison(s)")
        for _, row in mirna_results.iterrows():
            print(
                f"    {row['Comparison']}: log2FC={row['Log2_FC']:.3f}, q={row['Q_Value']:.3f}"
            )
else:
    print("❌ No significant differential expression results identified")

print(f"\n📊 Volcano plots saved to: {get_output_path('Volcano_Plots.png')}")


# 5. Machine Learning Models

Predictive modeling using logistic regression and random forest classifiers with cross-validation and performance evaluation.

In [ ]:
# Machine Learning Classification Models
print("🤖 MACHINE LEARNING MODELS")
print("=" * 50)

# Prepare data for machine learning
rq_cols = [f"RQ_{mirna}" for mirna in ANALYSIS_CONFIG["mirna_targets"]]
feature_columns = rq_cols + ANALYSIS_CONFIG["clinical_vars"]

# Remove any columns that don't exist in the dataset
available_features = [col for col in feature_columns if col in df.columns]
X = df[available_features]

print(f"📊 Features selected: {len(available_features)}")
print(f"Features: {available_features}")

# Store all model results
all_model_results = []

# Binary classification problems
binary_problems = [
    ("S", "P", "Healthy_vs_Periodontitis"),
    ("S", "G", "Healthy_vs_Gingivitis"),
    ("G", "P", "Gingivitis_vs_Periodontitis"),
]

for group1, group2, problem_name in binary_problems:
    print(f"\n🎯 {problem_name.replace('_', ' ')} Classification:")
    print("-" * 40)

    # Filter data for binary classification
    binary_data = df[df["GROUP"].isin([group1, group2])].copy()
    X_binary = binary_data[available_features]
    y_binary = binary_data["GROUP"]

    # Encode labels
    label_map = {group1: 0, group2: 1}
    y_binary_encoded = y_binary.map(label_map)

    print(
        f"  Sample sizes: {group1}={sum(y_binary_encoded == 0)}, {group2}={sum(y_binary_encoded == 1)}"
    )

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_binary,
        y_binary_encoded,
        test_size=ANALYSIS_CONFIG["test_size"],
        random_state=ANALYSIS_CONFIG["random_state"],
        stratify=y_binary_encoded,
    )

    # Feature scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Models to evaluate
    models = {
        "Logistic_Regression": LogisticRegression(
            random_state=ANALYSIS_CONFIG["random_state"]
        ),
        "Random_Forest": RandomForestClassifier(
            random_state=ANALYSIS_CONFIG["random_state"], n_estimators=100
        ),
    }

    problem_results = []

    for model_name, model in models.items():
        print(f"\n  🔮 {model_name.replace('_', ' ')} Model:")

        # Fit model
        if model_name == "Logistic_Regression":
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
            y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            y_pred_proba = model.predict_proba(X_test)[:, 1]

        # Calculate performance metrics
        accuracy = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_pred_proba)

        # Cross-validation
        cv_scores = cross_val_score(
            model,
            X_train_scaled if model_name == "Logistic_Regression" else X_train,
            y_train,
            cv=ANALYSIS_CONFIG["cv_folds"],
            scoring="roc_auc",
        )

        # Store results
        result = {
            "Problem": problem_name,
            "Model": model_name,
            "Accuracy": accuracy,
            "AUC": auc,
            "CV_Mean_AUC": cv_scores.mean(),
            "CV_Std_AUC": cv_scores.std(),
            "N_Train": len(X_train),
            "N_Test": len(X_test),
        }

        problem_results.append(result)
        all_model_results.append(result)

        print(f"    Accuracy: {accuracy:.3f}")
        print(f"    AUC: {auc:.3f}")
        print(f"    CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

        # Feature importance
        if model_name == "Random_Forest":
            feature_importance = model.feature_importances_
            feature_names = available_features

            # Create feature importance DataFrame
            importance_df = pd.DataFrame(
                {
                    "Feature": feature_names,
                    "Importance": feature_importance,
                    "Problem": problem_name,
                }
            ).sort_values("Importance", ascending=False)

            print(f"    Top 3 features:")
            for _, row in importance_df.head(3).iterrows():
                print(f"      {row['Feature']}: {row['Importance']:.3f}")

            # Save feature importance
            importance_filename = f"Feature_Importance_{problem_name}.csv"
            importance_df.to_csv(
                get_output_path(importance_filename, "tables"), index=False
            )

    print(f"  ✅ {problem_name} classification complete")

# Save all model results
model_results_df = pd.DataFrame(all_model_results)
model_results_df.to_csv(
    get_output_path("Model_Performance_Metrics.csv", "tables"), index=False
)

print(f"\n✅ Machine learning analysis complete!")
print(f"📊 {len(all_model_results)} model evaluations performed")
print(
    f"📊 Results saved to: {get_output_path('Model_Performance_Metrics.csv', 'tables')}"
)

# Display best performing models
print("\n🏆 BEST PERFORMING MODELS:")
print("=" * 50)
best_models = model_results_df.loc[model_results_df.groupby("Problem")["AUC"].idxmax()]
for _, row in best_models.iterrows():
    print(f"{row['Problem']}: {row['Model']} (AUC: {row['AUC']:.3f})")


In [ ]:
# ROC Curves and Confusion Matrices
print("\n📊 ROC CURVES AND CONFUSION MATRICES")
print("=" * 50)

# Create ROC curves for best models
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

plot_idx = 0

for group1, group2, problem_name in binary_problems:
    # Prepare data again for visualization
    binary_data = df[df["GROUP"].isin([group1, group2])].copy()
    X_binary = binary_data[available_features]
    y_binary = binary_data["GROUP"]

    label_map = {group1: 0, group2: 1}
    y_binary_encoded = y_binary.map(label_map)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_binary,
        y_binary_encoded,
        test_size=ANALYSIS_CONFIG["test_size"],
        random_state=ANALYSIS_CONFIG["random_state"],
        stratify=y_binary_encoded,
    )

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train both models
    models = {
        "Logistic Regression": LogisticRegression(
            random_state=ANALYSIS_CONFIG["random_state"]
        ),
        "Random Forest": RandomForestClassifier(
            random_state=ANALYSIS_CONFIG["random_state"], n_estimators=100
        ),
    }

    # ROC Curve
    ax_roc = axes[plot_idx]

    for model_name, model in models.items():
        if model_name == "Logistic Regression":
            model.fit(X_train_scaled, y_train)
            y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
        else:
            model.fit(X_train, y_train)
            y_pred_proba = model.predict_proba(X_test)[:, 1]

        fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
        auc = roc_auc_score(y_test, y_pred_proba)

        ax_roc.plot(fpr, tpr, label=f"{model_name} (AUC={auc:.3f})")

    ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.5)
    ax_roc.set_xlim([0, 1])
    ax_roc.set_ylim([0, 1])
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.set_title(f'ROC Curve: {problem_name.replace("_", " ")}')
    ax_roc.legend()
    ax_roc.grid(True, alpha=0.3)

    # Confusion Matrix (Random Forest)
    ax_cm = axes[plot_idx + 3]

    rf_model = RandomForestClassifier(
        random_state=ANALYSIS_CONFIG["random_state"], n_estimators=100
    )
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax_cm,
        xticklabels=[group1, group2],
        yticklabels=[group1, group2],
    )
    ax_cm.set_title(f'Confusion Matrix: {problem_name.replace("_", " ")}')
    ax_cm.set_xlabel("Predicted")
    ax_cm.set_ylabel("Actual")

    plot_idx += 1

plt.tight_layout()
plt.savefig(get_output_path("ROC_Curves.png"), dpi=300, bbox_inches="tight")
plt.show()

# Create separate confusion matrices plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, (group1, group2, problem_name) in enumerate(binary_problems):
    # Prepare data
    binary_data = df[df["GROUP"].isin([group1, group2])].copy()
    X_binary = binary_data[available_features]
    y_binary = binary_data["GROUP"]

    label_map = {group1: 0, group2: 1}
    y_binary_encoded = y_binary.map(label_map)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_binary,
        y_binary_encoded,
        test_size=ANALYSIS_CONFIG["test_size"],
        random_state=ANALYSIS_CONFIG["random_state"],
        stratify=y_binary_encoded,
    )

    # Train Random Forest
    rf_model = RandomForestClassifier(
        random_state=ANALYSIS_CONFIG["random_state"], n_estimators=100
    )
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)

    # Create confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=axes[i],
        xticklabels=[group1, group2],
        yticklabels=[group1, group2],
    )
    axes[i].set_title(f'{problem_name.replace("_", " ")}')
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")

plt.tight_layout()
plt.savefig(get_output_path("Confusion_Matrices.png"), dpi=300, bbox_inches="tight")
plt.show()

print(f"📊 ROC curves saved to: {get_output_path('ROC_Curves.png')}")
print(f"📊 Confusion matrices saved to: {get_output_path('Confusion_Matrices.png')}")


# 6. Dimensionality Reduction Analysis

PCA, t-SNE, and UMAP analysis for data visualization and clustering validation.

In [ ]:
# Dimensionality Reduction Analysis
print("📐 DIMENSIONALITY REDUCTION ANALYSIS")
print("=" * 50)

# Prepare data for dimensionality reduction
rq_cols = [f"RQ_{mirna}" for mirna in ANALYSIS_CONFIG["mirna_targets"]]
X_dr = df[rq_cols].copy()

# Standardize features
scaler = StandardScaler()
X_dr_scaled = scaler.fit_transform(X_dr)

# Color mapping for groups
color_map = {"S": "green", "G": "orange", "P": "red"}
colors = [color_map[group] for group in df["GROUP"]]

# Apply dimensionality reduction techniques
print("🔍 Applying dimensionality reduction techniques...")

# 1. PCA
print("  - Principal Component Analysis (PCA)")
pca = PCA(n_components=2, random_state=ANALYSIS_CONFIG["random_state"])
X_pca = pca.fit_transform(X_dr_scaled)

# 2. t-SNE
print("  - t-Distributed Stochastic Neighbor Embedding (t-SNE)")
tsne = TSNE(
    n_components=2,
    random_state=ANALYSIS_CONFIG["random_state"],
    perplexity=30,
    n_iter=1000,
)
X_tsne = tsne.fit_transform(X_dr_scaled)

# 3. UMAP
print("  - Uniform Manifold Approximation and Projection (UMAP)")
umap_reducer = umap.UMAP(n_components=2, random_state=ANALYSIS_CONFIG["random_state"])
X_umap = umap_reducer.fit_transform(X_dr_scaled)

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# PCA plot
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.7, s=60)
axes[0].set_title(
    f"PCA\n(PC1: {pca.explained_variance_ratio_[0]:.1%}, PC2: {pca.explained_variance_ratio_[1]:.1%})"
)
axes[0].set_xlabel("First Principal Component")
axes[0].set_ylabel("Second Principal Component")
axes[0].grid(True, alpha=0.3)

# t-SNE plot
axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=colors, alpha=0.7, s=60)
axes[1].set_title("t-SNE")
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")
axes[1].grid(True, alpha=0.3)

# UMAP plot
axes[2].scatter(X_umap[:, 0], X_umap[:, 1], c=colors, alpha=0.7, s=60)
axes[2].set_title("UMAP")
axes[2].set_xlabel("UMAP 1")
axes[2].set_ylabel("UMAP 2")
axes[2].grid(True, alpha=0.3)

# Add legend
legend_elements = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=color_map[group],
        markersize=10,
        label=f"Group {group}",
    )
    for group in ANALYSIS_CONFIG["groups"]
]
axes[0].legend(handles=legend_elements, loc="upper right")

plt.tight_layout()
plt.savefig(
    get_output_path("Dimensionality_Reduction.png"), dpi=300, bbox_inches="tight"
)
plt.show()

# PCA component analysis
print("\n📊 PCA COMPONENT ANALYSIS")
print("=" * 50)

pca_full = PCA(random_state=ANALYSIS_CONFIG["random_state"])
pca_full.fit(X_dr_scaled)

print(f"Explained variance ratio for first 6 components:")
for i, var_ratio in enumerate(pca_full.explained_variance_ratio_[:6]):
    print(f"  PC{i+1}: {var_ratio:.3f} ({var_ratio*100:.1f}%)")

print(
    f"Cumulative explained variance (first 6 components): {sum(pca_full.explained_variance_ratio_[:6]):.3f}"
)

# PCA loadings
loadings = pca_full.components_[:2].T  # First 2 components
loading_df = pd.DataFrame(loadings, columns=["PC1", "PC2"], index=rq_cols)
loading_df["Feature"] = loading_df.index
loading_df = loading_df.sort_values("PC1", key=abs, ascending=False)

print("\n📊 PCA Loadings (First 2 Components):")
print(loading_df.round(3))

# K-means clustering validation
print("\n🔍 CLUSTERING VALIDATION")
print("=" * 50)

# Apply K-means clustering
kmeans = KMeans(n_clusters=3, random_state=ANALYSIS_CONFIG["random_state"])
cluster_labels = kmeans.fit_predict(X_dr_scaled)

# Calculate adjusted rand score
ari = adjusted_rand_score(df["GROUP"].map({"S": 0, "G": 1, "P": 2}), cluster_labels)
print(f"Adjusted Rand Index: {ari:.3f}")

# Create cluster composition table
cluster_composition = pd.crosstab(df["GROUP"], cluster_labels, margins=True)
cluster_composition.index.name = "Actual_Group"
cluster_composition.columns.name = "Cluster"

print("\n📊 Cluster Composition:")
print(cluster_composition)

# Save cluster composition
cluster_composition.to_csv(get_output_path("Cluster_Composition.csv", "tables"))

# Visualize clustering results
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Original groups
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.7, s=60)
axes[0].set_title("Original Groups (PCA)")
axes[0].set_xlabel("First Principal Component")
axes[0].set_ylabel("Second Principal Component")
axes[0].grid(True, alpha=0.3)

# Cluster assignments
cluster_colors = ["purple", "cyan", "yellow"]
cluster_color_map = [cluster_colors[label] for label in cluster_labels]
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_color_map, alpha=0.7, s=60)
axes[1].set_title(f"K-means Clusters (ARI: {ari:.3f})")
axes[1].set_xlabel("First Principal Component")
axes[1].set_ylabel("Second Principal Component")
axes[1].grid(True, alpha=0.3)

# Add legends
legend_elements_groups = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=color_map[group],
        markersize=10,
        label=f"Group {group}",
    )
    for group in ANALYSIS_CONFIG["groups"]
]
axes[0].legend(handles=legend_elements_groups, loc="upper right")

legend_elements_clusters = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=cluster_colors[i],
        markersize=10,
        label=f"Cluster {i}",
    )
    for i in range(3)
]
axes[1].legend(handles=legend_elements_clusters, loc="upper right")

plt.tight_layout()
plt.savefig(get_output_path("Clustering_Results.png"), dpi=300, bbox_inches="tight")
plt.show()

print(f"\n✅ Dimensionality reduction analysis complete!")
print(f"📊 Plots saved to: {get_output_path('Dimensionality_Reduction.png')}")
print(f"📊 Clustering results saved to: {get_output_path('Clustering_Results.png')}")
print(
    f"📊 Cluster composition saved to: {get_output_path('Cluster_Composition.csv', 'tables')}"
)


# 7. Comprehensive Visualization Generation

Additional visualizations including boxplots, scatter plots, and feature importance analysis.

In [ ]:
# Comprehensive Visualization Generation
print("📊 COMPREHENSIVE VISUALIZATION GENERATION")
print("=" * 50)

# 1. Boxplots for pairwise comparisons
print("📦 Creating boxplots for pairwise comparisons...")

for group1, group2, comparison_name in comparison_pairs:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    comparison_data = df[df["GROUP"].isin([group1, group2])]

    for i, mirna in enumerate(ANALYSIS_CONFIG["mirna_targets"]):
        rq_col = f"RQ_{mirna}"

        sns.boxplot(
            data=comparison_data,
            x="GROUP",
            y=rq_col,
            ax=axes[i],
            palette="Set2",
            order=[group1, group2],
        )
        axes[i].set_title(f"{mirna} Expression")
        axes[i].set_ylabel("RQ Value")
        axes[i].set_xlabel("Group")
        axes[i].grid(True, alpha=0.3)

    plt.suptitle(f'{comparison_name.replace("_", " ")} Comparison', fontsize=16)
    plt.tight_layout()
    plt.savefig(
        get_output_path(f"Boxplots_{comparison_name}.png"), dpi=300, bbox_inches="tight"
    )
    plt.show()

# 2. Scatter plots for significant correlations
print("📊 Creating scatter plots for significant correlations...")

if significant_correlations:
    # Create scatter plots for top correlations
    n_plots = min(9, len(significant_correlations))  # Max 9 plots (3x3 grid)

    if n_plots > 0:
        fig, axes = plt.subplots(3, 3, figsize=(18, 18))
        axes = axes.flatten()

        for i in range(n_plots):
            corr = significant_correlations[i]
            var1, var2 = corr["Variable_1"], corr["Variable_2"]

            # Create scatter plot
            sns.scatterplot(
                data=df,
                x=var1,
                y=var2,
                hue="GROUP",
                ax=axes[i],
                palette="Set1",
                s=60,
                alpha=0.7,
            )
            axes[i].set_title(f'{var1} vs {var2}\n(r={corr["Correlation"]:.3f})')
            axes[i].grid(True, alpha=0.3)

        # Hide unused subplots
        for i in range(n_plots, 9):
            axes[i].set_visible(False)

        plt.suptitle("Significant Correlations", fontsize=16)
        plt.tight_layout()
        plt.savefig(
            get_output_path("Scatter_Plots_Significant_Correlations.png"),
            dpi=300,
            bbox_inches="tight",
        )
        plt.show()

# 3. Feature importance visualization
print("📊 Creating feature importance visualization...")

# Combine all feature importance results
all_importance_data = []
for problem_name in [comp[2] for comp in binary_problems]:
    try:
        importance_file = get_output_path(
            f"Feature_Importance_{problem_name}.csv", "tables"
        )
        if os.path.exists(importance_file):
            importance_data = pd.read_csv(importance_file)
            all_importance_data.append(importance_data)
    except:
        continue

if all_importance_data:
    combined_importance = pd.concat(all_importance_data, ignore_index=True)

    # Create feature importance plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for i, problem_name in enumerate([comp[2] for comp in binary_problems]):
        problem_data = combined_importance[
            combined_importance["Problem"] == problem_name
        ]

        if len(problem_data) > 0:
            problem_data = problem_data.sort_values("Importance", ascending=True)

            axes[i].barh(problem_data["Feature"], problem_data["Importance"])
            axes[i].set_title(f'{problem_name.replace("_", " ")}')
            axes[i].set_xlabel("Feature Importance")
            axes[i].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(get_output_path("Feature_Importance.png"), dpi=300, bbox_inches="tight")
    plt.show()

# 4. Clinical variable correlation heatmap
print("📊 Creating clinical variable correlation heatmap...")

# miRNA-clinical correlation matrix
mirna_clinical_vars = [
    f"RQ_{mirna}" for mirna in ANALYSIS_CONFIG["mirna_targets"]
] + ANALYSIS_CONFIG["clinical_vars"]
mirna_clinical_data = df[mirna_clinical_vars].select_dtypes(include=[np.number])

if len(mirna_clinical_data.columns) > 1:
    mirna_clinical_corr = mirna_clinical_data.corr()

    plt.figure(figsize=(12, 10))
    sns.heatmap(
        mirna_clinical_corr,
        annot=True,
        cmap="coolwarm",
        center=0,
        square=True,
        fmt=".2f",
        cbar_kws={"shrink": 0.8},
    )
    plt.title("miRNA-Clinical Variables Correlation Matrix")
    plt.tight_layout()
    plt.savefig(
        get_output_path("Correlation_Heatmap_miRNA_Clinical.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

# 5. Age and clinical scatter plots
print("📊 Creating age and clinical variable scatter plots...")

clinical_scatter_plots = [
    ("AGE", "pocket_depth"),
    ("AGE", "bleeding_on_probing"),
    ("plaque_index", "gingival_index"),
    ("plaque_index", "pocket_depth"),
    ("plaque_index", "bleeding_on_probing"),
    ("gingival_index", "pocket_depth"),
    ("gingival_index", "bleeding_on_probing"),
    ("pocket_depth", "bleeding_on_probing"),
]

for var1, var2 in clinical_scatter_plots:
    if var1 in df.columns and var2 in df.columns:
        plt.figure(figsize=(10, 6))
        sns.scatterplot(
            data=df, x=var1, y=var2, hue="GROUP", palette="Set1", s=60, alpha=0.7
        )
        plt.title(
            f'{var1.replace("_", " ").title()} vs {var2.replace("_", " ").title()}'
        )
        plt.xlabel(var1.replace("_", " ").title())
        plt.ylabel(var2.replace("_", " ").title())
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        # Format filename
        filename = f"Scatter_{var1.title()}_{var2.title()}.png"
        filename = filename.replace("_", "_")
        plt.savefig(get_output_path(filename), dpi=300, bbox_inches="tight")
        plt.show()

print("✅ Comprehensive visualization generation complete!")
print(f"📊 All visualizations saved to: {OUTPUT_DIRS['plots']}")

# Save processed data
print("\n💾 SAVING PROCESSED DATA")
print("=" * 50)

# Create comprehensive processed dataset
processed_data = df.copy()
processed_data.to_csv(get_output_path("Processed_Data.csv", "tables"), index=False)

print(f"📊 Processed data saved to: {get_output_path('Processed_Data.csv', 'tables')}")
print(f"📊 Shape: {processed_data.shape}")
print(f"📊 Columns: {len(processed_data.columns)}")


# 8. Results Export and Documentation

Final summary of results, file organization, and documentation updates.

In [ ]:
# Final Results Export and Summary
print("📋 FINAL RESULTS EXPORT AND SUMMARY")
print("=" * 50)

# Create comprehensive summary report
summary_report = {
    "analysis_timestamp": datetime.now().isoformat(),
    "dataset_info": {
        "file": DATA_FILE,
        "total_samples": len(df),
        "group_distribution": df["GROUP"].value_counts().to_dict(),
        "features_analyzed": len(ANALYSIS_CONFIG["mirna_targets"]),
        "clinical_variables": len(ANALYSIS_CONFIG["clinical_vars"]),
    },
    "statistical_results": {
        "omnibus_tests_performed": len(omnibus_results),
        "significant_omnibus_effects": (
            sum(omnibus_df["FDR_Significant"]) if len(omnibus_df) > 0 else 0
        ),
        "pairwise_comparisons": len(all_pairwise_results),
        "total_significant_results": (
            len(all_significant) if "all_significant" in locals() else 0
        ),
    },
    "machine_learning_results": {
        "models_evaluated": len(all_model_results),
        "best_auc_scores": {
            result["Problem"]: result["AUC"]
            for result in all_model_results
            if result["AUC"]
            == max(
                [
                    r["AUC"]
                    for r in all_model_results
                    if r["Problem"] == result["Problem"]
                ]
            )
        },
    },
    "output_files": {
        "plots_generated": len(
            [f for f in os.listdir(OUTPUT_DIRS["plots"]) if f.endswith(".png")]
        ),
        "tables_generated": len(
            [f for f in os.listdir(OUTPUT_DIRS["tables"]) if f.endswith(".csv")]
        ),
        "output_directory": BASE_OUTPUT_DIR,
    },
}

# Save summary report
with open(get_output_path("Analysis_Summary_Report.json", "tables"), "w") as f:
    json.dump(summary_report, f, indent=2)

print("📊 ANALYSIS SUMMARY")
print("=" * 50)
print(f"Total samples analyzed: {summary_report['dataset_info']['total_samples']}")
print(f"Group distribution: {summary_report['dataset_info']['group_distribution']}")
print(f"miRNAs analyzed: {summary_report['dataset_info']['features_analyzed']}")
print(
    f"Statistical tests performed: {summary_report['statistical_results']['omnibus_tests_performed']}"
)
print(
    f"Significant omnibus effects: {summary_report['statistical_results']['significant_omnibus_effects']}"
)
print(
    f"ML models evaluated: {summary_report['machine_learning_results']['models_evaluated']}"
)
print(f"Plots generated: {summary_report['output_files']['plots_generated']}")
print(f"Tables generated: {summary_report['output_files']['tables_generated']}")

# List all output files
print("\n📁 OUTPUT FILES GENERATED")
print("=" * 50)

print("📊 Tables:")
table_files = [f for f in os.listdir(OUTPUT_DIRS["tables"]) if f.endswith(".csv")]
for file in sorted(table_files):
    print(f"  - {file}")

print("\n📈 Plots:")
plot_files = [f for f in os.listdir(OUTPUT_DIRS["plots"]) if f.endswith(".png")]
for file in sorted(plot_files):
    print(f"  - {file}")

# Final recommendations
print("\n💡 ANALYTICAL RECOMMENDATIONS")
print("=" * 50)

recommendations = [
    "1. Consider validating key findings with additional reference genes",
    "2. Expand sample size for more robust statistical power",
    "3. Validate candidate biomarkers in independent cohorts",
    "4. Investigate biological pathways of significant miRNAs",
    "5. Consider longitudinal studies to assess temporal changes",
    "6. Explore integration with proteomic or metabolomic data",
]

for rec in recommendations:
    print(f"  {rec}")

# Technical notes
print("\n⚙️ TECHNICAL NOTES")
print("=" * 50)

technical_notes = [
    f"- Analysis performed with Python {sys.version.split()[0]}",
    f"- Random seed used: {ANALYSIS_CONFIG['random_state']}",
    f"- FDR correction method: {ANALYSIS_CONFIG['fdr_method']}",
    f"- Effect size threshold: {ANALYSIS_CONFIG['effect_size_threshold']}",
    f"- Cross-validation folds: {ANALYSIS_CONFIG['cv_folds']}",
    f"- Output directory: {BASE_OUTPUT_DIR}",
]

for note in technical_notes:
    print(f"  {note}")

print(f"\n✅ ANALYSIS COMPLETE!")
print(
    f"📊 Summary report saved to: {get_output_path('Analysis_Summary_Report.json', 'tables')}"
)
print(f"📁 All results saved to: {BASE_OUTPUT_DIR}")
print(f"🕐 Analysis completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Save final workspace state
final_workspace = {
    "variables_created": [var for var in dir() if not var.startswith("_")],
    "dataframe_shape": df.shape,
    "analysis_config": ANALYSIS_CONFIG,
    "output_directories": OUTPUT_DIRS,
}

with open(get_output_path("Final_Workspace_State.json", "tables"), "w") as f:
    json.dump(final_workspace, f, indent=2)

print(
    f"💾 Workspace state saved to: {get_output_path('Final_Workspace_State.json', 'tables')}"
)
